# SignalLens — Sentiment Baseline Comparison

ARCHITECTURE.md §39-40 / BUILD_PLAN.md Phase 17: compare our sentiment pipeline against standard baselines, reporting precision, recall, macro-F1, and a confusion matrix per class — never accuracy alone.

**Models compared**
1. **VADER** — lexicon/rule-based baseline (`vaderSentiment`).
2. **DistilBERT-SST2** — a standard small open sentiment model (`distilbert-base-uncased-finetuned-sst-2-english`). It is binary (positive/negative only); we adapt it to 3-class by treating low-confidence predictions as neutral, and call that adaptation out explicitly below.
3. **SignalLens pipeline** — `cardiffnlp/twitter-roberta-base-sentiment-latest`, the model `backend/app/services/sentiment_service.py` actually runs in production, applied at the same sentence/feedback-unit granularity.

**Ground-truth labeling — disclosure.** This environment has no Kaggle credentials and the demo corpus itself is template-generated (see `data/demo/generate_demo_dataset.py`), so the "hand-labeled sample" here is built by resampling that fixed set of hand-authored template sentences, each with a known, intentional ground-truth sentiment assigned when the template was written (e.g. every payment-failure template is unambiguously negative). This is a legitimate way to validate that the models agree with *known* ground truth, but it is **not** a substitute for independently human-labeled real production reviews — a real deployment should replace this cell with a genuinely independent 300–500 review label set (ARCHITECTURE.md §39).

In [ ]:
# Run once per environment. Safe to re-run.
%pip install -q vaderSentiment transformers torch scikit-learn matplotlib pandas

In [ ]:
import sys
from pathlib import Path
import random

sys.path.insert(0, str(Path("..").resolve() / ".." / "data" / "demo"))

from generate_demo_dataset import (
    PAYMENT_TEMPLATES,
    CRASH_TEMPLATES,
    OTP_TEMPLATES,
    PERFORMANCE_TEMPLATES,
    UI_TEMPLATES,
    SEARCH_TEMPLATES,
    SUPPORT_TEMPLATES,
    POSITIVE_TEMPLATES,
    NEUTRAL_TEMPLATES,
)

random.seed(7)

NEGATIVE_SOURCES = [
    PAYMENT_TEMPLATES,
    CRASH_TEMPLATES,
    OTP_TEMPLATES,
    PERFORMANCE_TEMPLATES,
    UI_TEMPLATES,
    SEARCH_TEMPLATES,
    SUPPORT_TEMPLATES,
]


def build_labeled_sample(n_negative=180, n_positive=140, n_neutral=80):
    rows = []
    for _ in range(n_negative):
        templates = random.choice(NEGATIVE_SOURCES)
        text = random.choice(templates).format(v="4.8.1")
        rows.append({"text": text, "true_label": "negative"})
    for _ in range(n_positive):
        rows.append({"text": random.choice(POSITIVE_TEMPLATES), "true_label": "positive"})
    for _ in range(n_neutral):
        rows.append({"text": random.choice(NEUTRAL_TEMPLATES), "true_label": "neutral"})
    random.shuffle(rows)
    return rows


labeled_sample = build_labeled_sample()
print(f"Labeled sample size: {len(labeled_sample)}")
from collections import Counter
print(Counter(r["true_label"] for r in labeled_sample))

## 1. VADER baseline

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

vader = SentimentIntensityAnalyzer()


def vader_predict(text: str) -> str:
    compound = vader.polarity_scores(text)["compound"]
    if compound >= 0.05:
        return "positive"
    if compound <= -0.05:
        return "negative"
    return "neutral"


vader_predictions = [vader_predict(r["text"]) for r in labeled_sample]

## 2. DistilBERT-SST2 baseline (binary, adapted to 3-class)

SST2 only outputs POSITIVE/NEGATIVE. We treat predictions with confidence below 0.75 as neutral — a reasonable but inexact adaptation, which is exactly the kind of limitation a 2-class model has against a 3-class problem; it is expected to underperform on the neutral class.

In [ ]:
from transformers import pipeline

distilbert_pipe = pipeline(
    "sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english"
)


def distilbert_predict_batch(texts: list[str]) -> list[str]:
    results = distilbert_pipe(texts, truncation=True, batch_size=32)
    predictions = []
    for r in results:
        if r["score"] < 0.75:
            predictions.append("neutral")
        else:
            predictions.append("positive" if r["label"] == "POSITIVE" else "negative")
    return predictions


distilbert_predictions = distilbert_predict_batch([r["text"] for r in labeled_sample])

## 3. SignalLens pipeline (CardiffNLP RoBERTa, as run in production)

In [ ]:
sys.path.insert(0, str(Path("..").resolve() / ".." / "backend"))
from app.services.sentiment_service import analyze_sentiment_batch

signallens_results = analyze_sentiment_batch([r["text"] for r in labeled_sample])
signallens_predictions = [label for label, _confidence in signallens_results]

## 4. Scoring — precision / recall / macro-F1 / confusion matrix (never accuracy alone)

In [ ]:
import pandas as pd
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

LABELS = ["negative", "neutral", "positive"]
y_true = [r["true_label"] for r in labeled_sample]

models = {
    "VADER": vader_predictions,
    "DistilBERT-SST2": distilbert_predictions,
    "SignalLens (RoBERTa)": signallens_predictions,
}

summary_rows = []
for name, y_pred in models.items():
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=LABELS, average="macro", zero_division=0
    )
    summary_rows.append(
        {"model": name, "macro_precision": round(precision, 3), "macro_recall": round(recall, 3), "macro_f1": round(f1, 3)}
    )

summary_df = pd.DataFrame(summary_rows).sort_values("macro_f1", ascending=False)
summary_df

In [ ]:
for name, y_pred in models.items():
    print(f"\n=== {name} ===")
    print(classification_report(y_true, y_pred, labels=LABELS, zero_division=0))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 4))
for ax, (name, y_pred) in zip(axes, models.items()):
    cm = confusion_matrix(y_true, y_pred, labels=LABELS)
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(LABELS)))
    ax.set_yticks(range(len(LABELS)))
    ax.set_xticklabels(LABELS, rotation=45)
    ax.set_yticklabels(LABELS)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(name)
    for i in range(len(LABELS)):
        for j in range(len(LABELS)):
            ax.text(j, i, cm[i, j], ha="center", va="center")
plt.tight_layout()
plt.show()

## Takeaways

- VADER is a fast, dependency-light lexicon baseline but has no learned understanding of context (sarcasm, negation chains, domain-specific phrases like "OTP never arrived").
- DistilBERT-SST2 is a strong *binary* classifier but structurally cannot represent neutral sentiment — its neutral-class recall is expected to be weak here because we only approximate neutral via a confidence threshold.
- The SignalLens pipeline (CardiffNLP RoBERTa, applied per feedback-unit rather than per whole review) is natively 3-class and tuned on social-media-style short text, which is structurally closer to app review sentences than either baseline.
- This notebook is a methodology demonstration, not a final accuracy claim — re-run it against a genuinely independently labeled sample of real reviews before citing these numbers externally.